Working with pandas data frames

In [1]:
import pandas as pd

In [37]:
DATA_PATH = "...\\APAN5400\\APAN5400\\data"
cb_sample = pd.read_csv(f"{DATA_PATH}\\cb_sample.csv")

In [22]:
cb_sample.shape
cb_sample.columns

Index(['name', 'city', 'region', 'country', 'category_list',
       'category_groups_list', 'cb_url', 'rank', 'domain'],
      dtype='str')

In [38]:
cb_sample[['name','region']].head()

,name,region
0,Eventige,New York
1,Eventik,Distrito Federal
2,Eventil,NaN
3,"Eventiles, Inc.",Washington
4,Eventilio,Delhi


In [39]:
print(cb_sample[['region']].isna().sum())
print(cb_sample[['region']].isna())
print(cb_sample[(cb_sample[['region']].isna()).any(axis=1)] )

#only drop rows where 'region' is NaN
cb_sample_clean = cb_sample.dropna(subset=['region'])
cb_sample_clean[['region']].isna().sum()

region    25
dtype: int64
    region
0    False
1    False
2     True
3    False
4    False
..     ...
94   False
95    True
96    True
97   False
98   False

[99 rows x 1 columns]
                        name city region country  \
2                    Eventil  NaN    NaN     NaN   
5                 Eventility  NaN    NaN     NaN   
9        eventinformation.dk  NaN    NaN      DK   
12                 Eventions  NaN    NaN     NaN   
14                   Eventis  NaN    NaN     NaN   
20                  Eventizy  NaN    NaN      IN   
21                 Eventjams  NaN    NaN     NaN   
29  EventKlang Entertainment  NaN    NaN      DE   
31                EVENTKUTTA  NaN    NaN     NaN   
32                  Eventleo  NaN    NaN     NaN   
35                Eventlocus  NaN    NaN     NaN   
36                EventLolli  NaN    NaN     NaN   
37                 EventLoud  NaN    NaN     NaN   
48              Eventmetrics  NaN    NaN     NaN   
50                Eventninja  NaN    Na

region    0
dtype: int64

In [41]:
#checking for duplicates in dataframe
print(cb_sample_clean[cb_sample_clean.duplicated(keep=False)].sort_values(by=cb_sample_clean.columns.tolist()))
print(cb_sample_clean.shape)


                  name      city         region country  \
40          EventMagic   Oakland     California      US   
87          EventMagic   Oakland     California      US   
42           EventMama    Dublin         Dublin      IE   
89           EventMama    Dublin         Dublin      IE   
43          EventMania    Palmas      Tocantins      BR   
90          EventMania    Palmas      Tocantins      BR   
44  EventMap Solutions   Belfast        Belfast      GB   
91  EventMap Solutions   Belfast        Belfast      GB   
46          EventMates  New York       New York      US   
93          EventMates  New York       New York      US   
47         EventMethod    Austin          Texas      US   
94         EventMethod    Austin          Texas      US   
39         Eventmag.ru    Moscow    Moscow City      RU   
86         Eventmag.ru    Moscow    Moscow City      RU   
41       Eventmaker.io     Paris  Île-de-France      FR   
88       Eventmaker.io     Paris  Île-de-France      FR 

In [42]:
#Now drop the first duplicate
cb_sample_no_dups = cb_sample_clean.drop_duplicates(keep='first')
print(cb_sample_no_dups.shape)

(63, 9)


In [45]:
#Create a JSON file out of the cleaned dataframe
cb_sample_no_dups.to_json("...\\APAN5400\\APAN5400\\data\\cb_sample_no_dups.json", orient="records")
print(cb_sample_no_dups.shape)

(63, 9)


Flask

In [13]:
# Uncomment the line below to install Flask package
#pip install flask
#!pip install -U flask

In [11]:
from pathlib import Path

import pandas as pd
from flask import Flask, request, render_template

#Summary
# This code sets up a Flask web application that allows users to search for movie titles in a CSV file.
# It reads the "movies.csv" file into a pandas DataFrame, and if the user submits a search term via the form,
# it filters the DataFrame for titles containing the term (case-insensitive) and returns the corresponding genres.
# The results are then rendered in the "my-form.html" template.

module_folder = Path.cwd()
if not (module_folder / "my-form.html").exists():
    module_folder = Path.cwd() / "Module3"

df = pd.read_csv(module_folder / "movies.csv")

app = Flask(__name__, template_folder=str(module_folder))

@app.route("/", methods=["GET", "POST"])
def search_titles():
    results = None
    message = ""

    if request.method == "POST":
        term = request.form.get("term", "").strip()
        if term:
            matches = df[df["title"].str.contains(term, case=False, na=False)]
            results = matches["genre"].to_list()

    return render_template("my-form.html", results=results, message=message)

app.run(host="127.0.0.1", port=5002, debug=False, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5002
Press CTRL+C to quit
127.0.0.1 - - [27/Sep/2026 10:18:37] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [27/Sep/2026 10:18:41] "POST / HTTP/1.1" 200 -


In [12]:
import json
import pandas as pd
PATH = "...\\APAN5400\\APAN5400\\data\\webzio_apple.json"

In [13]:

json_data=open(PATH).readlines()
newsfeeds = []
for line in json_data:
    newsfeeds.append(json.loads(line))
print(len(newsfeeds))

10800


In [14]:
df = pd.DataFrame(newsfeeds)

An Interactive App for webzio_apple.json

In [16]:
from flask import Flask, request, render_template
module_folder = Path.cwd()
if not (module_folder / "my-form-news.html").exists():
    module_folder = Path.cwd() / "Module3"

#Summary
# This code sets up a Flask web application that allows users to search for news articles in a CSV file.
# It reads the "cb_sample.csv" file into a pandas DataFrame, and if the user submits a search term via the form,
# it filters the DataFrame for titles containing the term (case-insensitive) and returns the corresponding details.
# The results are then rendered in the "my-form-news.html" template.

app = Flask("Interactive App", template_folder=str(module_folder))

@app.route('/')
def my_form():
    return render_template("my-form-news.html", results=None)

@app.route('/', methods=['POST'])
def my_form_post():
    val = request.form['userinput'] #User input value
    matches = df[df['title'].str.contains(val, case=False, na=False)] #exact-match string search

    results = [
        {
            "title": row["title"],
            "site": row.get("site", ""),
            "published": row.get("published", ""),
            "text": (row.get("text", "") or "")[:300],
            "url": row.get("url", ""),
        }
        for _, row in matches.iterrows()
    ]

    return render_template("my-form-news.html", results=results)

In [17]:

app.run(host="127.0.0.1", port=5001, debug=False, use_reloader=False)

 * Serving Flask app 'Interactive App'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [27/Sep/2026 10:21:17] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [27/Sep/2026 10:21:17] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [27/Sep/2026 10:21:21] "POST / HTTP/1.1" 200 -


Using Fask Jonsify

In [55]:
from pathlib import Path

import pandas as pd
from flask import Flask, jsonify

DATA_PATH = Path.cwd().parent / "data" / "companies.csv"
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd() / "data" / "companies.csv"

company = pd.read_csv(DATA_PATH)

app = Flask(__name__)

@app.route("/")
def get_company():
    return jsonify(company.to_dict(orient="records"))

if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\samardeen\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
